[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xpopdev/nltk-tokenize-rs-20260817/blob/main/benchmark_speed_test.ipynb)

# `ported_lib` vs `nltk` — Speed Test (small → large)

**Rust-backed drop-in for `nltk.tokenize` — same outputs, faster.**  
This notebook times the original `nltk` against the `ported_lib` (PyO3 + maturin) port across batch sizes from 1 string to corpora scale. Every cell is runnable top-to-bottom; re-run after `maturin develop --release`.

| Scale | What we measure | Why it matters |
|---|---|---|
| single call | `word_tokenize` / `sent_tokenize` per-call latency | hot-path tokenization |
| 10 – 100 texts | `*_batch` (rayon) vs sequential loop | request / document batches |
| 1k – 10k texts | throughput plateau | corpus pipelines |
| Gutenberg / long doc | MB-scale `word_tokenize` + `sent_tokenize` | full-book processing |

In [ ]:
# ── Colab one-click setup ──
# On Colab this cell installs the Rust toolchain + builds ported_lib.
# Locally it's a no-op (ported_lib already installed via maturin).
import importlib.util, sys
if importlib.util.find_spec("ported_lib") is None and "google.colab" in sys.modules:
    print("Colab detected — installing Rust + building ported_lib … (1-2 min)")
    import subprocess, textwrap
    # Rust
    subprocess.run("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable", shell=True, check=False)
    # maturin + nltk + build
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "maturin", "nltk", "matplotlib", "pandas"], check=False)
    subprocess.run([sys.executable, "-m", "maturin", "develop", "--release"], check=False)
    print("done — re-import below will pick up ported_lib")
else:
    print("skip colab setup (local or already installed)")


## 0 — Setup & imports

In [ ]:
import importlib, sys, time, statistics, textwrap, platform
from pathlib import Path

# --- ensure ported_lib is importable (maturin develop --release) ---
try:
    import ported_lib
    print(f"ported_lib  {ported_lib.__file__}")
except ImportError as e:
    print(f"ported_lib not installed — run: pip install maturin && maturin develop --release\n{e}")
    raise SystemExit(1)

import nltk
from nltk.tokenize import word_tokenize as nltk_word_tokenize, sent_tokenize as nltk_sent_tokenize
from nltk.tokenize.casual import TweetTokenizer as NLTKTweetTokenizer
from nltk.tokenize.toktok import ToktokTokenizer as NLTKToktok

print(f"nltk        {nltk.__version__}  @ {Path(nltk.__file__).parent}")
print(f"python      {platform.python_version()}  ({platform.machine()})")
print(f"gpu avail?  {ported_lib.is_gpu_available()}  — {ported_lib.gpu_info()[:120]}")

# warm up Rust LazyLocks so init cost doesn't pollute the first timing
ported_lib.gpu_warmup()
_ = ported_lib.word_tokenize("warm up.")
_ = ported_lib.sent_tokenize("Warm up. Second sentence.")
_ = nltk_word_tokenize("warm up.")
_ = nltk_sent_tokenize("Warm up. Second sentence.")
print("warmup done ✓")

## 1 — Correctness sanity check (must match before speed means anything)

In [ ]:
cases = [
    "Hello, world. It costs $5.00.",
    "Mr. Smith went home. He left.",
    "Good muffins cost $3.88 in New York. Please buy me two of them.",
    "The U.S. is large. U.S. troops moved.",  # ortho break vs no-break
    'He said "Stop." She left.',
    "J. R. R. Tolkien wrote LOTR.",
]

ok = True
for text in cases:
    a, b = nltk_word_tokenize(text), ported_lib.word_tokenize(text)
    c, d = nltk_sent_tokenize(text), ported_lib.sent_tokenize(text)
    if a != b or c != d:
        ok = False
        print(f"MISMATCH {text!r}\n  word  nltk={a}  ported={b}\n  sent  nltk={c}  ported={d}")
print("word/sent parity ✓" if ok else "parity FAILED — fix correctness before benchmarking")

## 2 — Helpers (median-of-7, with warmup)

In [ ]:
def time_fn(fn, arg, reps: int = 1, runs: int = 7, warmup: int = 2) -> float:
    """Median seconds per call of fn(arg) — warmup excluded, median of `runs`."""
    for _ in range(warmup):
        try: fn(arg)
        except Exception: pass
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        for _ in range(reps):
            fn(arg)
        times.append((time.perf_counter() - t0) / reps)
    return statistics.median(times)

def fmt_us(s: float) -> str:
    return f"{s*1e6:,.1f} µs"

def fmt_ms(s: float) -> str:
    return f"{s*1e3:,.2f} ms"

SENTENCES = [
    "Good muffins cost $3.88 in New York. Please buy me two of them.",
    "Mr. Smith went to Washington. He saw Dr. Jones and Mrs. Brown.",
    "Hello, world! This is a test. Visit https://example.com today :)",
    "The quick brown fox jumps over the lazy dog. Pack my box with five dozen liquor jugs.",
    "She can't've gone — it's too late. Won't you come? I'd've helped.",
    "Toktok handles | pipes, [brackets], and 3.14 numbers.",
]

TWEETS = [
    "Just saw @elonmusk launch another rocket! #space https://spacex.com 🚀",
    "Can't believe it!!! Sooooo coooool :) :) :) #amazing",
]

print("helpers ready ✓")

## 3 — Single-call latency (the headline number)

Same cases as `scripts/benchmark.py` (`--reps 200`, median-of-7) — plus a long doc for throughput.

In [ ]:
REPS = 200
long_word = "Hello, world. " * 700          # ~10k chars
long_sent = "Mr. Smith went to Washington. He saw Dr. Jones. " * 100

cases_word = [("hello world",), ("Hello, world. It costs $5.",), ((long_word,),)][-1:]  # smoke + long
# actually reuse gen_matrix_inputs cases if available
try:
    sys.path.insert(0, str(Path("scripts").resolve()))
    from compare_outputs import FUNCTION_PAIRS, _try_import_pairs
    from gen_matrix_inputs import cases_for
    _try_import_pairs()
    HAVE_MATRIX = bool(FUNCTION_PAIRS)
except Exception as e:
    HAVE_MATRIX = False
    print(f"matrix helpers unavailable ({e}) — using ad-hoc cases")

def bench_pair(label, orig, ported, cases):
    args = [(c.args, c.kwargs) for c in cases] if HAVE_MATRIX else [((cases[0],), {})]
    # add long case for heavy fns
    if label in ("word_tokenize", "sent_tokenize"):
        extra = long_sent if label == "sent_tokenize" else long_word
        args = args + [((extra,), {})]
    def run_orig():
        for a, kw in args: orig(*a, **kw)
    def run_port():
        for a, kw in args: ported(*a, **kw)
    # warmup
    for _ in range(2):
        try: run_orig()
        except: pass
        try: run_port()
        except: pass
    import statistics as _s, time as _t
    to = _s.median([(_t.perf_counter(), [run_orig() for _ in range(REPS)], _t.perf_counter())[2] - (_t.perf_counter(),)[0] for _ in range(1)])
    # simpler: reuse time_fn
    to = time_fn(run_orig, None, reps=REPS)
    tp = time_fn(run_port, None, reps=REPS)
    return to, tp, to/tp if tp else float('inf')

# direct, readable version without the matrix indirection:
def bench_simple(label, orig_fn, port_fn, texts):
    def do_orig(_):
        for t in texts: orig_fn(t)
    def do_port(_):
        for t in texts: port_fn(t)
    to = time_fn(do_orig, None, reps=REPS)
    tp = time_fn(do_port, None, reps=REPS)
    calls = len(texts)
    return to/calls, tp/calls, (to/tp) if tp else float('inf')

bench_texts = ["Hello, world. It costs $5.00.", "Mr. Smith went home. He left.", long_word]
pairs = [
    ("word_tokenize",      lambda t: nltk_word_tokenize(t),        lambda t: ported_lib.word_tokenize(t)),
    ("sent_tokenize",      lambda t: nltk_sent_tokenize(t),        lambda t: ported_lib.sent_tokenize(t)),
    ("casual_tokenize",    lambda t: NLTKTweetTokenizer().tokenize(t), lambda t: ported_lib.casual_tokenize_py(t)),
    ("toktok_tokenize",    lambda t: NLTKToktok().tokenize(t),     lambda t: ported_lib.toktok_tokenize_py(t)),
]
rows = []
for label, fo, fp in pairs:
    o, p, s = bench_simple(label, fo, fp, bench_texts)
    rows.append((label, o, p, s))
    print(f"{label:18s}  nltk {fmt_us(o):>12s}   ported {fmt_us(p):>12s}   {s:5.1f}×")

try:
    import pandas as pd
    df = pd.DataFrame(rows, columns=["function","nltk (s)","ported (s)","speedup"])
    df["nltk (µs)"] = (df["nltk (s)"]*1e6).round(1)
    df["ported (µs)"] = (df["ported (s)"]*1e6).round(1)
    df["speedup"] = df["speedup"].round(2)
    display(df[["function","nltk (µs)","ported (µs)","speedup"]])
except ImportError:
    pass

### Full `benchmark_report.md` table (from last CI run, for reference)

Re-run `python scripts/benchmark.py --reps 200` to refresh locally.

In [ ]:
try:
    print(Path("benchmark_report.md").read_text())
except FileNotFoundError:
    print("benchmark_report.md not found — run scripts/benchmark.py first")

## 4 — Batch scaling: 1 → 10 → 100 → 1k → 10k texts

Compares three paths on the same input list:
- **seq** — Python loop over single-string API (what users do today)
- **batch** — `ported_lib.*_batch` (rayon, CPU-parallel)
- **batch_gpu / rayon** — same `*_batch` alias (wgpu removed in 1.0.0; name kept for compat)

We sweep powers of ten so the log-log plot shows where parallelism kicks in (≥64 items). `PAR_CHUNK` = `available_parallelism()`.

In [ ]:
import json

SIZES = [1, 10, 100, 1_000, 10_000]

def make_texts(base, n):
    return (base * ((n // len(base)) + 1))[:n]

def bench_batch_sweep(label, base, seq_fn, batch_fn, sizes=SIZES, reps=5):
    rows = []
    for n in sizes:
        texts = make_texts(base, n)
        t_seq   = time_fn(lambda ts: [seq_fn(t) for t in ts], texts, reps=reps)
        t_batch = time_fn(batch_fn, texts, reps=reps)
        rows.append({
            "n": n,
            "seq_ms": t_seq*1e3, "batch_ms": t_batch*1e3,
            "seq_vs_batch": t_seq/t_batch if t_batch else float('inf'),
            "tok_per_s_seq": n / t_seq if t_seq else 0,
            "tok_per_s_batch": n / t_batch if t_batch else 0,
        })
        print(f"  n={n:5d}  seq {fmt_ms(t_seq):>10s}  batch {fmt_ms(t_batch):>10s}  {t_seq/t_batch:5.1f}×  "
              f"({n/t_seq:,.0f} vs {n/t_batch:,.0f} docs/s)")
    return rows

print("=== word_tokenize ===")
word_rows = bench_batch_sweep("word_tokenize", SENTENCES,
    lambda t: ported_lib.word_tokenize(t),
    lambda ts: ported_lib.word_tokenize_batch(ts))

print("\n=== sent_tokenize ===")
sent_rows = bench_batch_sweep("sent_tokenize", SENTENCES,
    lambda t: ported_lib.sent_tokenize(t),
    lambda ts: ported_lib.sent_tokenize_batch(ts))

print("\n=== casual / TweetTokenizer ===")
try:
    nltk_tweet = NLTKTweetTokenizer()
    casual_rows = bench_batch_sweep("casual", TWEETS,
        lambda t: nltk_tweet.tokenize(t),
        lambda ts: [ported_lib.casual_tokenize_py(t) for t in ts])
    # also ported single vs batch
    print("\n=== casual (ported single vs ported batch) ===")
    casual_ported_rows = bench_batch_sweep("casual_ported", TWEETS,
        lambda t: ported_lib.casual_tokenize_py(t),
        lambda ts: ported_lib.casual_tokenize_batch_gpu(ts, True, False, False, True))
except Exception as e:
    print(f"casual sweep skipped: {e}")
    casual_rows = []

# stash for plotting next cell
_batch_results = {"word": word_rows, "sent": sent_rows, "casual": casual_rows}

### Batch plot (log–log — falls back to table if matplotlib missing)

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for label, rows in [("word_tokenize", word_rows), ("sent_tokenize", sent_rows)]:
        ns = [r["n"] for r in rows]
        axes[0].loglog(ns, [r["seq_ms"] for r in rows], marker="o", label=f"{label} seq")
        axes[0].loglog(ns, [r["batch_ms"] for r in rows], marker="s", label=f"{label} batch")
        axes[1].semilogx(ns, [r["seq_vs_batch"] for r in rows], marker="o", label=label)
    axes[0].set_title("Batch latency (ms) — lower is better")
    axes[0].set_xlabel("batch size (texts)"); axes[0].set_ylabel("ms")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].set_title("Speedup: seq → batch (rayon)")
    axes[1].set_xlabel("batch size"); axes[1].set_ylabel("×")
    axes[1].axhline(1, color="grey", ls="--", alpha=0.5)
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed — table fallback (pip install matplotlib to plot)")
    try:
        import pandas as pd
        for k, rows in _batch_results.items():
            if rows:
                print(f"\n{k}")
                display(pd.DataFrame(rows).round(2))
    except ImportError:
        import json; print(json.dumps(_batch_results, indent=2))

## 5 — Large document (Gutenberg-scale)

One big string (joined paragraphs) — measures sustained throughput, not just per-call overhead. Falls back to synthetic text if `nltk.corpus.gutenberg` isn't downloaded.

In [ ]:
LARGE_TEXT = None
src_label = "synthetic"
try:
    from nltk.corpus import gutenberg
    try:
        nltk.data.find("corpora/gutenberg")
    except LookupError:
        print("downloading gutenberg (~5 MB)…")
        nltk.download("gutenberg", quiet=True)
    raws = [gutenberg.raw(fid) for fid in gutenberg.fileids()]
    raws = [t for t in raws if t.strip()]
    LARGE_TEXT = "\n\n".join(raws)
    src_label = f"gutenberg ({len(raws)} files)"
except Exception as e:
    print(f"gutenberg unavailable ({e}) — using synthetic ~2 MB text")
    LARGE_TEXT = ("Good muffins cost $3.88 in New York. Mr. Smith went to Washington. " * 20000)
    src_label = "synthetic"

size_mb = len(LARGE_TEXT.encode()) / 1e6
print(f"source: {src_label}  —  {len(LARGE_TEXT):,} chars  ({size_mb:.2f} MB)")

# single-pass large-doc timings
for label, fn_nltk, fn_port in [
    ("word_tokenize", lambda t: nltk_word_tokenize(t), lambda t: ported_lib.word_tokenize(t)),
    ("sent_tokenize", lambda t: nltk_sent_tokenize(t), lambda t: ported_lib.sent_tokenize(t)),
]:
    # verify parity on a slice before timing the whole thing
    probe = LARGE_TEXT[:2000]
    assert fn_nltk(probe) == fn_port(probe), f"{label} parity failed on Gutenberg probe"
    t_n = time_fn(fn_nltk, LARGE_TEXT, reps=1, runs=5)
    t_p = time_fn(fn_port, LARGE_TEXT, reps=1, runs=5)
    mb_s_n = size_mb / t_n if t_n else 0
    mb_s_p = size_mb / t_p if t_p else 0
    print(f"{label:15s}  nltk {fmt_ms(t_n):>10s} ({mb_s_n:5.1f} MB/s)  "
          f"ported {fmt_ms(t_p):>10s} ({mb_s_p:5.1f} MB/s)  {t_n/t_p:5.1f}×")

print("\nTip: for the full Gutenberg bench script, run  python scripts/gutenberg_bench.py  "
      "(also available as the gutenberg-bench workflow)")

## 6 — All tokenizers at a glance (ported vs nltk where applicable)

Covers the full family — skips those `nltk` doesn't expose as a one-liner (e.g. `sonority`, `legality`).

In [ ]:
import ast

# (label, nltk_fn, ported_fn, sample_text)
sample = "Hello, world. It costs $5.00. Visit https://example.com @user #tag :)"
more = [
    ("word_tokenize",       lambda t: nltk_word_tokenize(t),                    lambda t: ported_lib.word_tokenize(t), sample),
    ("wordpunct",           lambda t: nltk.wordpunct_tokenize(t),               lambda t: ported_lib.wordpunct_tokenize_py(t), sample),
    ("whitespace",          lambda t: nltk.WhitespaceTokenizer().tokenize(t),   lambda t: ported_lib.whitespace_tokenize_py(t), sample),
    ("regexp \\s+",        lambda t: nltk.RegexpTokenizer(r"\s+", gaps=True).tokenize(t), lambda t: ported_lib.regexp_tokenize(t, r"\s+", gaps=True), sample),
    ("toktok",              lambda t: NLTKToktok().tokenize(t),                 lambda t: ported_lib.toktok_tokenize_py(t), sample),
    ("Treebank detok",      lambda t: nltk.TreebankWordDetokenizer().detokenize(t.split()), lambda t: ported_lib.detokenize(t.split()), sample),
]

for label, fn_n, fn_p, txt in more:
    try:
        a, b = fn_n(txt), fn_p(txt)
        ok = "✓" if a == b else "≠ (known deviation — see PLAN §8)"
        print(f"{label:18s} {ok:40s}  nltk={str(a)[:60]}")
    except Exception as e:
        print(f"{label:18s} skipped ({e})")

print("\nBenchmark these with REPS=1000 for tighter medians:")
REPS2 = 1000
for label, fn_n, fn_p, txt in more[:4]:
    try:
        tn = time_fn(fn_n, txt, reps=REPS2)
        tp = time_fn(fn_p, txt, reps=REPS2)
        print(f"  {label:18s}  {tn*1e6:7.1f} µs → {tp*1e6:7.1f} µs  {tn/tp:5.1f}×")
    except Exception as e:
        print(f"  {label:18s}  skipped ({e})")

## 7 — Export summary (optional)

Writes `notebook_benchmark_summary.json` for CI / PR comments.

In [ ]:
import json, datetime
summary = {
    "generated": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "python": platform.python_version(),
    "nltk": nltk.__version__,
    "ported_lib": getattr(ported_lib, "__version__", "1.0.0"),
    "single_call": [{"fn": r[0], "nltk_us": r[1]*1e6, "ported_us": r[2]*1e6, "speedup": r[3]} for r in rows] if 'rows' in globals() else [],
    "batch": {k: v for k, v in _batch_results.items()} if '_batch_results' in globals() else {},
    "large_doc": {"source": src_label if 'src_label' in globals() else "?", "mb": round(size_mb, 3) if 'size_mb' in globals() else 0},
}
Path("notebook_benchmark_summary.json").write_text(json.dumps(summary, indent=2))
print(Path("notebook_benchmark_summary.json").read_text()[:2000])